# PGExplainer -- Edge-Level Fraud Explanation for the Frozen PNA Classifier

**Pipeline stage:** `preprocessing.ipynb` -> `aml_graph_construction.ipynb` ->
`train_baseline_comparison.ipynb` -> **this notebook** -> LLM evidence-locked prompting

**Goal:** given a transaction the frozen PNA classifier has flagged as laundering,
produce a structured, evidence-locked explanation of *why* -- a ranked list of the
other transactions (edges) in its local neighborhood that most shaped the model's
decision, plus a feature-level breakdown of the flagged transaction's own fields.
This structured JSON is what the LLM stage will be constrained to cite from --
it never sees raw graph structure or invents evidence of its own.

**Decisions this notebook implements (agreed before any code was written):**

1. **Implementation approach:** custom PGExplainer, not `torch_geometric.explain.PGExplainer`
   off the shelf. The classifier's split `encode()`/`score()` design, the per-batch
   `augment_subgraph()` step (self-loops + reverse edges applied *outside* the model),
   and the target edge's own features bypassing message passing entirely in `score()`
   don't match the single-`forward()` contract the built-in `Explainer` API expects.
   We do reuse one low-level PyG utility (`set_masks`/`clear_masks` from
   `torch_geometric.explain.algorithm.utils`) for the actual mask injection, since
   that hooks into `MessagePassing.propagate()` at the base-class level and works
   identically regardless of `PNAConv`'s specific `forward()` signature.

2. **Explanation target:** for a flagged transaction, output = (a) its own raw
   features (always included, un-masked -- `score()` never routes the target edge
   through message passing) + a feature-level Integrated Gradients breakdown of
   those features (Decision 2/Scenario B), and (b) a ranked list of the *other*
   transactions in its k-hop computational subgraph that shaped `h_src`/`h_dst`,
   scored by a trained PGExplainer MLP (edge-level attribution only -- no IG on
   neighbors, see Section 13).

3. **Forward/reverse merge + self-loop exclusion:** every real transaction enters
   the augmented subgraph twice (original + `is_reverse=1` copy). Their importance
   scores are merged by `max()` into one transaction-level score before ranking.
   Synthetic self-loops added by `augment_subgraph` (all-zero `edge_attr`, distinct
   from a genuine self-transaction which carries `is_self_loop=1` in its real
   `edge_attr`) are excluded entirely -- they aren't real transactions.

4. **Training population:** the explainer MLP is trained/tuned/evaluated on
   *predicted-positive* edges only (post-threshold, not ground-truth positives) --
   matching the actual deployment use case, where explanations are only ever
   requested for transactions the model already flagged. Phase-scoping is
   preserved throughout (train predicted-positives use `train_base_ei/ea`, val use
   `val_base_ei/ea`, test use `test_base_ei/ea` -- test touched once, at the end).

5. **Faithfulness metric:** Fidelity+ / Fidelity- (probability-drop when removing
   vs. keeping only the explanation), with a random-k baseline for comparison.
   Top-k chosen on val via a characterization score, confirmed once on test.

6. **Feature-level attribution (Scenario B):** Integrated Gradients, target
   transaction only (not neighbors -- the neighbor GNN path is expensive to run IG
   through and its importance is already covered by the PGExplainer edge score).
   Baseline = 0 for continuous (a trained `StandardScaler`'s 0 *is* the train-set
   mean) and binary features. Categorical columns (`Payment Format_enc` etc.) are
   reported as plain facts, no IG score -- interpolating a categorical code has no
   meaning, and the LLM stage doesn't need a numeric weight to use a categorical
   fact correctly.

**A note on what this notebook does NOT do:** it never re-trains or fine-tunes the
PNA classifier itself (`best_model_pna.pt` stays frozen throughout), and it never
uses `pattern_diagnostics.csv` / ground-truth pattern labels as a training signal --
only as an optional, offline sanity check on the demo outputs at the very end
(Section 15), consistent with the project's existing rule that pattern-type labels
are diagnostics only.


## 1. Imports & config

In [ ]:
# --- Imports ---
import gc
import json
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import PNAConv
from torch_geometric.utils import add_remaining_self_loops
from torch_geometric.explain.algorithm.utils import set_masks, clear_masks

from sklearn.metrics import precision_recall_curve, f1_score
from sklearn.metrics import auc as sk_auc
DATA_DIR = Path("/home/jovyan/AML_Project/NoteBooks/Data/Bassam Data")
assert DATA_DIR.exists(), f"{DATA_DIR} not found."

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


In [ ]:
# --- Config ---
# Classifier-side settings mirror train_baseline_comparison.ipynb exactly -- these
# are NOT tunable here, they describe the frozen model we're loading.
NUM_NEIGHBORS_INFER = [10, 5]      # must match BEST_HP["num_neighbors"] used to
                                    # produce best_model_pna.pt -- overwritten below
                                    # once we load best_hyperparams_pna.json
INFER_BATCH_SIZE = 16384

# --- Explainer-side settings (new, tunable) ---
EXPLAINER_HIDDEN = 64              # MLP hidden width
EXPLAINER_EPOCHS = 10              # small MLP, few epochs -- watch val Fidelity,
                                    # raise if it's still improving
EXPLAINER_LR = 3e-3
EXPLAINER_BATCH_SIZE = 256         # number of TARGET transactions per batch (each
                                    # one pulls its own k-hop subgraph -- this is
                                    # not the same kind of batch as classifier
                                    # training, keep it small)
MASK_SPARSITY_COEF = 0.03          # penalizes the mean mask value -- encourages a
                                    # sparse (few important edges), not diffuse, mask.
                                    # FIX (first training run): loss_pred is trivially
                                    # minimized by mask=1 everywhere (no masking at all
                                    # exactly reproduces the original prediction), so
                                    # sparsity is the ONLY force pushing masks toward 0
                                    # -- at 5e-4 it was ~70x weaker than the converged
                                    # loss_pred (~0.036) and every mask saturated at 1.0.
                                    # 0.03 is a starting point, not a tuned value -- watch
                                    # the resulting mean mask value and raise further if
                                    # importance scores still don't differentiate.
MASK_ENTROPY_COEF = 1e-3           # penalizes masks that sit near 0.5 (encourages
                                    # confident 0/1-ish decisions, standard PGExplainer reg.)
                                    # Left unchanged for this run -- isolate the effect of
                                    # the sparsity fix first before touching this too.
TEMPERATURE = 1.0                  # Gumbel/concrete relaxation temperature (kept
                                    # fixed here; anneal 5.0 -> 1.0 across epochs if
                                    # masks look too noisy in practice)

TOP_K_CANDIDATES = [3, 5, 10, 15]  # swept on val to pick the final k (Decision 5)
IG_STEPS = 50                      # Integrated Gradients interpolation steps

RANDOM_SEED = 2000
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Config loaded.")


## 2. Load `graph_data.pt`, metadata, encoders/scalers, and the human-readable lookup

Everything here is read-only -- nothing in this section is fit on any data, it's
all artifacts already produced and saved by the earlier notebooks.


In [ ]:
data = torch.load(DATA_DIR / "graph_data.pt", weights_only=False)
print(data)

with open(DATA_DIR / "metadata.json") as f:
    metadata = json.load(f)

edge_cat_start = data.edge_cat_start if hasattr(data, "edge_cat_start") else metadata["edge_cat_start"]
EDGE_FEATURE_COLS = metadata["edge_feature_cols"]
CAT_COLS = ["Payment Format", "Payment Currency", "Receiving Currency"]
vocab_sizes = [metadata["vocab_sizes"][c] for c in CAT_COLS]

n_nodes = data.x.size(0)
in_dim = data.x.size(1)
print(f"edge_cat_start={edge_cat_start}  n_nodes={n_nodes:,}  in_dim={in_dim}")
print(f"EDGE_FEATURE_COLS ({len(EDGE_FEATURE_COLS)}): {EDGE_FEATURE_COLS}")

# Column index of "is_self_loop" within edge_attr -- needed in Section 6 to tell a
# genuine self-transaction (real edge_attr, is_self_loop=1) apart from a synthetic
# self-loop added by augment_subgraph (all-zero edge_attr, is_self_loop reads 0
# only because everything was zero-filled, not because it's a real flag).
IS_SELF_LOOP_COL = EDGE_FEATURE_COLS.index("is_self_loop")
print(f"is_self_loop column index: {IS_SELF_LOOP_COL}")


In [ ]:
# --- Encoders / scalers (for decoding model-ready features back to human-readable
# values in the final JSON output -- the open item flagged from a prior session) ---
with open(DATA_DIR / "encoders.pkl", "rb") as f:
    cat_encoders = pickle.load(f)          # {col_name: {original_value: code}}

with open(DATA_DIR / "edge_scaler.pkl", "rb") as f:
    edge_scaler = pickle.load(f)           # fit on CONTINUOUS_EDGE_COLS, train-only

CONTINUOUS_EDGE_COLS = ["log_amount_paid", "log_amount_received", "log_amount_ratio",
                         "log_hours_since_last_tx", "log_tx_count_cumulative"]
# EDGE_FEATURE_COLS carries these as "scaled_<col>" -- map scaled name -> raw name
SCALED_TO_RAW = {f"scaled_{c}": c for c in CONTINUOUS_EDGE_COLS}

# Reverse category maps: code -> original string, per categorical column
cat_decoders = {col: {v: k for k, v in enc.items()} for col, enc in cat_encoders.items()}

# --- Human-readable transaction lookup + transaction_id <-> edge_idx mapping ---
transactions_lookup = pd.read_csv(DATA_DIR / "transactions_lookup.csv")
transaction_ids = np.load(DATA_DIR / "transaction_ids.npy", allow_pickle=True)  # (E,), same row order as edge_index/edge_attr

txn_id_to_edge_idx = {tid: i for i, tid in enumerate(transaction_ids)}
edge_idx_to_txn_id = {i: tid for tid, i in txn_id_to_edge_idx.items()}

# transaction_id -> which split it belongs to (train/val/test) -- needed later to
# pick the right phase-scoped base graph inside explain_transaction()
txn_id_to_split = dict(zip(transactions_lookup["transaction_id"], transactions_lookup["split"]))

print(f"transactions_lookup: {transactions_lookup.shape}")
print(f"{len(txn_id_to_edge_idx):,} transaction_id -> edge_idx mappings built.")


## 3. Rebuild the frozen PNA `EdgeClassifier`

**These class definitions are copied verbatim from `train_baseline_comparison.ipynb`
(Sections 4/5) -- PNA only, since that's the winning/saved architecture.** If that
notebook's `EdgeEncoder`/`NodeEncoderPNA`/`EdgeClassifier`/`augment_subgraph` ever
change, this cell has to be updated to match, or `best_model_pna.pt`'s state_dict
won't load. Nothing here is retrained -- we load the saved weights and freeze
every parameter immediately after.


In [ ]:
class EdgeEncoder(nn.Module):
    def __init__(self, edge_cat_start, vocab_sizes, emb_dim=8, out_dim=32):
        super().__init__()
        self.edge_cat_start = edge_cat_start
        n_continuous = edge_cat_start + 1  # +1 for is_reverse
        self.cont_proj = nn.Linear(n_continuous, out_dim)
        self.cat_embeddings = nn.ModuleList([nn.Embedding(v, emb_dim) for v in vocab_sizes])
        self.out_proj = nn.Linear(out_dim + emb_dim * len(vocab_sizes), out_dim)

    def forward(self, edge_attr, is_reverse=None):
        if is_reverse is None:
            is_reverse = torch.zeros(edge_attr.size(0), 1, device=edge_attr.device, dtype=edge_attr.dtype)
        cont = torch.cat([edge_attr[:, :self.edge_cat_start], is_reverse], dim=1)
        cont_out = F.relu(self.cont_proj(cont))
        cat = edge_attr[:, self.edge_cat_start:].long()
        cat_out = torch.cat([emb(cat[:, i]) for i, emb in enumerate(self.cat_embeddings)], dim=1)
        return self.out_proj(torch.cat([cont_out, cat_out], dim=1))


class NodeEncoderPNA(nn.Module):
    def __init__(self, in_dim, hidden_dim, deg, n_layers=2, dropout=0.2, edge_dim=32, towers=2):
        super().__init__()
        aggregators = ["mean", "min", "max", "std"]
        scalers = ["identity", "amplification", "attenuation"]
        conv_kwargs = dict(aggregators=aggregators, scalers=scalers, deg=deg,
                            edge_dim=edge_dim, towers=towers, pre_layers=1, post_layers=1)
        self.convs = nn.ModuleList([PNAConv(in_dim, hidden_dim, **conv_kwargs)])
        for _ in range(n_layers - 1):
            self.convs.append(PNAConv(hidden_dim, hidden_dim, **conv_kwargs))
        self.dropout = dropout

    def forward(self, x, edge_index, edge_embedding):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index, edge_embedding)
            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return x


class EdgeClassifier(nn.Module):
    def __init__(self, model_name, in_dim, edge_cat_start, vocab_sizes,
                 hidden_dim=32, n_layers=2, dropout=0.2, deg=None, pna_towers=2):
        super().__init__()
        assert model_name == "pna", "This notebook only rebuilds the PNA winner."
        self.model_name = model_name
        self.hidden_dim = hidden_dim
        self.uses_edge_attr_in_mp = True

        self.edge_encoder = EdgeEncoder(edge_cat_start, vocab_sizes, emb_dim=8, out_dim=hidden_dim)
        assert deg is not None, "PNA requires a train-only degree histogram"
        self.node_encoder = NodeEncoderPNA(in_dim, hidden_dim, deg, n_layers, dropout,
                                            edge_dim=hidden_dim, towers=pna_towers)

        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def encode(self, x, edge_index, edge_attr, is_reverse=None):
        edge_emb = self.edge_encoder(edge_attr, is_reverse)
        return self.node_encoder(x, edge_index, edge_emb)

    def score(self, h_src, h_dst, target_edge_attr):
        target_edge_emb = self.edge_encoder(target_edge_attr)
        z = torch.cat([h_src, h_dst, target_edge_emb], dim=1)
        return self.head(z).squeeze(-1)


def augment_subgraph(edge_index, edge_attr, num_nodes):
    # Verbatim from train_baseline_comparison.ipynb Section 3 -- see that notebook
    # for the full explanation. Returns (edge_index, edge_attr, is_reverse).
    ei = edge_index
    ea = edge_attr
    is_reverse = torch.zeros(ei.size(1), 1, dtype=torch.float)

    non_self_loop = ei[0] != ei[1]
    rev_ei = ei[:, non_self_loop].flip(0)
    rev_ea = ea[non_self_loop]
    rev_is_reverse = torch.ones(rev_ei.size(1), 1, dtype=torch.float)

    ei = torch.cat([ei, rev_ei], dim=1)
    ea = torch.cat([ea, rev_ea], dim=0)
    is_reverse = torch.cat([is_reverse, rev_is_reverse], dim=0)

    n_before = ei.size(1)
    ei, ea = add_remaining_self_loops(ei, ea, fill_value=0.0, num_nodes=num_nodes)
    n_added = ei.size(1) - n_before
    if n_added > 0:
        is_reverse = torch.cat([is_reverse, torch.zeros(n_added, 1, dtype=torch.float)], dim=0)

    return ei, ea, is_reverse


def augment_subgraph_traceable(edge_index, edge_attr, e_id, num_nodes):
    # Same logic as augment_subgraph, but threads the ORIGINAL global edge id
    # (sub.e_id from NeighborLoader) through the reverse-copy and self-loop steps
    # too, so every row in the returned edge set can be traced back to a real
    # transaction_id (or flagged as synthetic with e_id = -1).
    #
    # BUG FIX (found while debugging the Section 6 assertion): add_remaining_self_loops
    # does NOT simply append new self-loops after the existing edges in place -- it
    # REMOVES every existing self-loop from its original position first
    # (internally: mask = edge_index[0] != edge_index[1]), then rebuilds ONE
    # self-loop block of size num_nodes, ordered by node index (not by original
    # edge order), copying the real edge_attr back in for nodes that already had a
    # self-loop and using fill_value for the rest. A naive "assume the last
    # (new_count - old_count) rows are synthetic" concat -- which is what this
    # function originally did, and what augment_subgraph (Section 3, copied
    # verbatim from train_baseline_comparison.ipynb) still does for is_reverse --
    # desyncs eid/is_reverse from edge_attr whenever a node in the subgraph has a
    # genuine self-transaction (is_self_loop=1), because that real self-loop gets
    # moved into the middle of the rebuilt block, not left in place. Fixed here by
    # replicating add_remaining_self_loops' exact internal logic by hand, so eid
    # (and is_reverse) can be threaded through it correctly instead of guessing
    # where rows landed afterward. See Section 6b for a diagnostic on how often
    # this actually matters for the classifier''s own (unfixed) augment_subgraph.
    ei = edge_index
    ea = edge_attr
    eid = e_id.clone()
    is_reverse = torch.zeros(ei.size(1), 1, dtype=torch.float)

    non_self_loop = ei[0] != ei[1]
    rev_ei = ei[:, non_self_loop].flip(0)
    rev_ea = ea[non_self_loop]
    rev_eid = eid[non_self_loop]              # same transaction_id as its forward copy
    rev_is_reverse = torch.ones(rev_ei.size(1), 1, dtype=torch.float)

    ei = torch.cat([ei, rev_ei], dim=1)
    ea = torch.cat([ea, rev_ea], dim=0)
    eid = torch.cat([eid, rev_eid], dim=0)
    is_reverse = torch.cat([is_reverse, rev_is_reverse], dim=0)

    # --- Hand-written equivalent of add_remaining_self_loops(fill_value=0.0),
    # extended to also carry eid and is_reverse through correctly ---
    mask = ei[0] != ei[1]                        # True for non-self-loop rows
    self_loop_mask = ~mask
    existing_loop_nodes = ei[0, self_loop_mask]   # node index of each pre-existing self-loop
    existing_loop_ea = ea[self_loop_mask]
    existing_loop_eid = eid[self_loop_mask]
    existing_loop_is_rev = is_reverse[self_loop_mask]  # always 0 for real self-loops

    loop_index = torch.arange(num_nodes, dtype=ei.dtype, device=ei.device)
    loop_ei = loop_index.unsqueeze(0).repeat(2, 1)
    loop_ea = ea.new_full((num_nodes,) + ea.size()[1:], 0.0)          # fill_value=0.0
    loop_eid = torch.full((num_nodes,), -1, dtype=eid.dtype, device=eid.device)
    loop_is_reverse = torch.zeros(num_nodes, 1, dtype=torch.float)

    loop_ea[existing_loop_nodes] = existing_loop_ea
    loop_eid[existing_loop_nodes] = existing_loop_eid
    loop_is_reverse[existing_loop_nodes] = existing_loop_is_rev

    final_ei = torch.cat([ei[:, mask], loop_ei], dim=1)
    final_ea = torch.cat([ea[mask], loop_ea], dim=0)
    final_eid = torch.cat([eid[mask], loop_eid], dim=0)
    final_is_reverse = torch.cat([is_reverse[mask], loop_is_reverse], dim=0)

    return final_ei, final_ea, final_is_reverse, final_eid


In [ ]:
# --- Load the frozen checkpoint ---
with open(DATA_DIR / "best_hyperparams_pna.json") as f:
    best_hp = json.load(f)
NUM_NEIGHBORS_INFER = best_hp["num_neighbors"]
print(f"Loaded best_hyperparams_pna.json -- num_neighbors={NUM_NEIGHBORS_INFER}, "
      f"hidden_dim={best_hp['hidden_dim']}, n_layers={best_hp['n_layers']}")

checkpoint = torch.load(DATA_DIR / "best_model_pna.pt", weights_only=False)
assert checkpoint["model_name"] == "pna"
pna_deg = checkpoint["pna_deg_histogram"]

model = EdgeClassifier(
    "pna", in_dim, edge_cat_start, vocab_sizes,
    hidden_dim=checkpoint["hyperparameters"]["hidden_dim"],
    n_layers=checkpoint["hyperparameters"]["n_layers"],
    dropout=checkpoint["hyperparameters"]["dropout"],
    deg=pna_deg,
).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Freeze -- this notebook trains only the small explainer MLP defined in Section 7,
# the classifier itself never gets a gradient update from here on.
for p in model.parameters():
    p.requires_grad_(False)

print(f"Loaded frozen PNA classifier from seed {checkpoint['seed']}. Params frozen: "
      f"{sum(p.numel() for p in model.parameters()):,}")


## 4. Recompute the locked decision threshold (Decision 4)

`best_model_pna.pt` saves the winning seed's weights but **not** the val-locked
threshold picked for it in Section 10 of `train_baseline_comparison.ipynb` --
that value only ever lived in an in-memory variable during that run. The
computation is deterministic (a function of the frozen weights + the val split),
so we reproduce it here rather than re-deriving it from scratch: same
`compute_node_embeddings_for` / `evaluate` logic, run once, on val only.


In [ ]:
val_base_mask = data.train_mask | data.val_mask
val_base_ei = data.edge_index[:, val_base_mask]
val_base_ea = data.edge_attr[val_base_mask]

train_base_ei = data.edge_index[:, data.train_mask]
train_base_ea = data.edge_attr[data.train_mask]

test_base_ei = data.edge_index
test_base_ea = data.edge_attr

BASE_EDGES_BY_PHASE = {
    "train": (train_base_ei, train_base_ea),
    "val":   (val_base_ei, val_base_ea),
    "test":  (test_base_ei, test_base_ea),
}


def _local_lookup(seed_ids, query_ids):
    sorted_ids, sort_perm = torch.sort(seed_ids)
    pos_in_sorted = torch.searchsorted(sorted_ids, query_ids)
    return sort_perm[pos_in_sorted]


@torch.no_grad()
def compute_node_embeddings_for(base_ei, base_ea, target_idx, num_neighbors, batch_size):
    # Verbatim logic from train_baseline_comparison.ipynb Section 6, using the
    # frozen `model` loaded above.
    target_nodes = torch.unique(torch.cat([
        data.edge_index[0, target_idx], data.edge_index[1, target_idx]
    ]))
    base_graph = Data(x=data.x, edge_index=base_ei, edge_attr=base_ea, num_nodes=data.x.size(0))
    loader = NeighborLoader(base_graph, num_neighbors=num_neighbors, input_nodes=target_nodes,
                             batch_size=batch_size, shuffle=False)

    all_h = torch.zeros(data.x.size(0), model.hidden_dim)
    for sub in loader:
        aug_ei, aug_ea, aug_is_rev = augment_subgraph(sub.edge_index, sub.edge_attr, sub.num_nodes)
        x_dev = sub.x.to(DEVICE)
        aug_ei, aug_ea, aug_is_rev = aug_ei.to(DEVICE), aug_ea.to(DEVICE), aug_is_rev.to(DEVICE)
        h = model.encode(x_dev, aug_ei, aug_ea, aug_is_rev)
        all_h[sub.n_id[:sub.batch_size]] = h[:sub.batch_size].cpu()
        del sub, aug_ei, aug_ea, aug_is_rev, h
    del base_graph
    return all_h


@torch.no_grad()
def score_edges(base_ei, base_ea, target_idx, num_neighbors, batch_size, threshold=None):
    h = compute_node_embeddings_for(base_ei, base_ea, target_idx, num_neighbors, batch_size).to(DEVICE)
    src = data.edge_index[0, target_idx].to(DEVICE)
    dst = data.edge_index[1, target_idx].to(DEVICE)
    ea = data.edge_attr[target_idx].to(DEVICE)
    y = data.y[target_idx].float()

    logits_chunks = []
    for start in range(0, target_idx.size(0), batch_size):
        end = start + batch_size
        logits_chunks.append(model.score(h[src[start:end]], h[dst[start:end]], ea[start:end]).cpu())
    logits = torch.cat(logits_chunks)
    probs = torch.sigmoid(logits)

    y_np, probs_np = y.numpy().astype(int), probs.numpy()
    precision, recall, pr_thresholds = precision_recall_curve(y_np, probs_np)
    pr_auc = sk_auc(recall, precision)

    if threshold is None:
        f1_curve = np.divide(2 * precision * recall, precision + recall,
                              out=np.zeros_like(precision), where=(precision + recall) > 0)
        best_idx = int(np.argmax(f1_curve[:-1])) if len(pr_thresholds) > 0 else 0
        threshold = float(pr_thresholds[best_idx]) if len(pr_thresholds) > 0 else 0.5
        f1 = float(f1_curve[best_idx]) if len(pr_thresholds) > 0 else 0.0
    else:
        preds_np = (probs_np >= threshold).astype(int)
        f1 = f1_score(y_np, preds_np, pos_label=1, zero_division=0)

    del h
    return {"f1": f1, "pr_auc": pr_auc, "probs": probs, "threshold": threshold}


val_metrics = score_edges(val_base_ei, val_base_ea, data.val_mask.nonzero(as_tuple=True)[0],
                           NUM_NEIGHBORS_INFER, INFER_BATCH_SIZE)
LOCKED_THRESHOLD = val_metrics["threshold"]
print(f"Recomputed val-locked threshold = {LOCKED_THRESHOLD:.4f}  "
      f"(val F1={val_metrics['f1']:.4f}, matches Section 10's seed {checkpoint['seed']} run if close to "
      f"baseline_comparison_summary.json's reported test performance)")


### 4b. Seeding verification (run first, cheap)

`get_target_context` (Section 10) reseeds `torch.manual_seed(target_edge_idx)` right
before calling `NeighborLoader`, so the same transaction always resolves to the same
sampled subgraph (Section 10's markdown explains why this matters -- without it, the
same transaction produced two different model probabilities on this environment in an
earlier run). Neighbor sampling here goes through a compiled extension
(`torch.ops.torch_sparse.neighbor_sample`), and PyTorch's own docs note that custom
operators aren't always guaranteed to respect `torch.manual_seed()`. This cell checks
that assumption directly, in under a second, self-contained (no dependency on
`predicted_positive` or Section 10's helpers, so it runs this early) -- if it fails,
better to find out now than after a multi-hour training run.


In [ ]:
def _seed_check_subgraph(target_edge_idx, seed):
    src = data.edge_index[0, target_edge_idx].item()
    dst = data.edge_index[1, target_edge_idx].item()
    seed_nodes = torch.tensor([src, dst], dtype=torch.long)
    base_graph = Data(x=data.x, edge_index=train_base_ei, edge_attr=train_base_ea, num_nodes=data.x.size(0))
    torch.manual_seed(seed)
    np.random.seed(seed % (2**32 - 1))
    loader = NeighborLoader(base_graph, num_neighbors=NUM_NEIGHBORS_INFER, input_nodes=seed_nodes,
                             batch_size=seed_nodes.size(0), shuffle=False)
    sub = next(iter(loader))
    return sub.edge_index.clone()


_check_idx = data.train_mask.nonzero(as_tuple=True)[0][0].item()
_ei_a = _seed_check_subgraph(_check_idx, _check_idx)
_ei_b = _seed_check_subgraph(_check_idx, _check_idx)
SEED_CHECK_PASSED = torch.equal(_ei_a, _ei_b)

print(f"Seeding verification (torch.manual_seed controls NeighborLoader sampling): "
      f"{'PASS' if SEED_CHECK_PASSED else 'FAIL'}")
if not SEED_CHECK_PASSED:
    print("*** WARNING: get_target_context()'s determinism fix (Section 10) does NOT "
          "work on this environment's torch_sparse backend -- explanations may still "
          "vary between calls for the same transaction. The training run below will "
          "still proceed (it doesn't depend on this), but do not trust per-transaction "
          "Fidelity/importance reproducibility in Sections 14/15 until this is fixed. "
          "Flag this exact message to Claude for a backend-specific alternative. ***")


## 5. Predicted-positive edge sets per phase (Decision 4)

The explainer is trained/tuned/evaluated only on transactions the **frozen model
itself** flags as fraud (`prob >= LOCKED_THRESHOLD`), scored within each phase's
own message-passing scope -- not on ground-truth positives. This scores every edge
in each mask once, which is the same order of cost as Section 10's final test run
in the baseline notebook (that touched all 31.9M edges once already), so it's
within the budget this environment has already handled -- but for `train_mask`
(22.3M edges) specifically, run this as a detached background job
(`nohup python -u ... &`, same workaround as before) rather than in an
interactive cell if the kernel-idle-culling issue shows up again.


In [ ]:
def get_predicted_positive_edges(phase, threshold=LOCKED_THRESHOLD):
    base_ei, base_ea = BASE_EDGES_BY_PHASE[phase]
    mask = getattr(data, f"{phase}_mask")
    target_idx = mask.nonzero(as_tuple=True)[0]
    t0 = time.time()
    metrics = score_edges(base_ei, base_ea, target_idx, NUM_NEIGHBORS_INFER,
                           INFER_BATCH_SIZE, threshold=threshold)
    probs = metrics["probs"]
    pred_pos_local = (probs >= threshold).nonzero(as_tuple=True)[0]
    pred_pos_global = target_idx[pred_pos_local]
    n_true_pos = int(data.y[target_idx].sum().item())
    print(f"[{phase}] scored {target_idx.numel():,} edges in {time.time()-t0:.1f}s -- "
          f"{pred_pos_global.numel():,} predicted positive (of {n_true_pos:,} true positive), "
          f"F1={metrics['f1']:.4f}")
    return pred_pos_global


predicted_positive = {}
for phase in ["train", "val", "test"]:
    predicted_positive[phase] = get_predicted_positive_edges(phase)

# Save so this expensive scoring pass doesn't need to be repeated if the kernel
# restarts partway through the rest of the notebook.
torch.save(predicted_positive, DATA_DIR / "predicted_positive_edges.pt")
print("Saved predicted_positive_edges.pt")


## 6. Edge-id resolution, forward/reverse merge, and self-loop exclusion (Decision 3)

`sub.e_id` (from `NeighborLoader`) gives the **global index into `data.edge_index` /
`data.edge_attr` / `transaction_ids`** for every edge PyG actually sampled --
this is what makes it possible to disambiguate parallel edges (multiple real
transactions between the same account pair), which a plain `(src, dst)` match
cannot do. This assert is here specifically to fail loudly, early, if a different
PyG version doesn't expose it, rather than silently producing wrong transaction_id
attributions later.


In [ ]:
def _sanity_check_e_id_available():
    probe_nodes = data.edge_index[0, :64]
    probe_graph = Data(x=data.x, edge_index=train_base_ei, edge_attr=train_base_ea,
                        num_nodes=data.x.size(0))
    probe_loader = NeighborLoader(probe_graph, num_neighbors=[5, 5], input_nodes=probe_nodes,
                                   batch_size=probe_nodes.size(0), shuffle=False)
    probe_sub = next(iter(probe_loader))
    assert hasattr(probe_sub, "e_id"), (
        "NeighborLoader did not return `e_id` on this PyG version -- Section 6's "
        "transaction-id tracing (and everything downstream: merge, ranking, the "
        "final evidence JSON) depends on it. Check the installed torch_geometric "
        "version / sampling backend (pyg-lib vs torch-sparse) before proceeding."
    )
    print(f"e_id available -- sanity check passed. Example values: {probe_sub.e_id[:5].tolist()}")


_sanity_check_e_id_available()


In [ ]:
def merge_and_filter(edge_index, e_id, importance_raw, edge_attr):
    """Collapse forward/reverse duplicates of the same transaction (max of their
    scores) and drop synthetic self-loops, returning one row per real transaction.

    importance_raw: 1D tensor, same length as edge_index.size(1) / e_id -- the raw
                     (already sigmoid-applied, in [0,1]) explainer score per edge
                     in the augmented subgraph.
    Returns: dict[int edge_idx (global, into data.edge_index)] -> float importance
    """
    is_synthetic_self_loop = (e_id == -1)

    # A genuine self-transaction (account paying itself) is NOT synthetic -- it has
    # e_id >= 0 and its real edge_attr, distinguishable via the is_self_loop column
    # (only meaningful when e_id >= 0; a synthetic row's edge_attr is all zero and
    # e_id is already -1, so this second check is a belt-and-suspenders assertion).
    if is_synthetic_self_loop.any():
        synth_ea = edge_attr[is_synthetic_self_loop]
        assert torch.allclose(synth_ea, torch.zeros_like(synth_ea)), \
            "A row flagged synthetic (e_id=-1) has non-zero edge_attr -- augment_subgraph_traceable bug."

    keep = ~is_synthetic_self_loop
    kept_eid = e_id[keep].tolist()
    kept_scores = importance_raw[keep].tolist()

    merged = {}
    for eid, score in zip(kept_eid, kept_scores):
        if eid not in merged or score > merged[eid]:
            merged[eid] = score
    return merged


### 6b. Diagnostic: quantifying the same bug in the classifier's `augment_subgraph`

Section 6 fixed an `e_id` desync in `augment_subgraph_traceable` caused by
`add_remaining_self_loops` reordering existing self-loops instead of leaving them
in place. **The exact same reordering affects `is_reverse` in the original,
already-used `augment_subgraph`** (Section 3, copied verbatim from
`train_baseline_comparison.ipynb`) -- it doesn't corrupt the message-passing
structure itself (`edge_index`/`edge_attr` are unaffected), only the `is_reverse`
bit fed into `EdgeEncoder` for rows near where a genuine self-transaction sat in
the array. This cell measures how often that actually happens on real data, so
the decision to (or not to) revisit `best_model_pna.pt` is based on a number, not
a guess.


In [ ]:
def _is_reverse_buggy(ei, ea, num_nodes):
    # Exact replica of augment_subgraph's is_reverse handling (Section 3), for comparison.
    is_reverse = torch.zeros(ei.size(1), 1, dtype=torch.float)
    non_self_loop = ei[0] != ei[1]
    rev_ei = ei[:, non_self_loop].flip(0)
    rev_ea = ea[non_self_loop]
    rev_is_reverse = torch.ones(rev_ei.size(1), 1, dtype=torch.float)
    ei2 = torch.cat([ei, rev_ei], dim=1)
    ea2 = torch.cat([ea, rev_ea], dim=0)
    is_reverse2 = torch.cat([is_reverse, rev_is_reverse], dim=0)
    n_before = ei2.size(1)
    ei3, ea3 = add_remaining_self_loops(ei2, ea2, fill_value=0.0, num_nodes=num_nodes)
    n_added = ei3.size(1) - n_before
    if n_added > 0:
        is_reverse2 = torch.cat([is_reverse2, torch.zeros(n_added, 1, dtype=torch.float)], dim=0)
    return ei3, is_reverse2


def _is_reverse_correct(ei, ea, num_nodes):
    # Section 6's fixed logic, is_reverse only (no eid needed for this comparison).
    is_reverse = torch.zeros(ei.size(1), 1, dtype=torch.float)
    non_self_loop = ei[0] != ei[1]
    rev_ei = ei[:, non_self_loop].flip(0)
    rev_is_reverse = torch.ones(rev_ei.size(1), 1, dtype=torch.float)
    ei2 = torch.cat([ei, rev_ei], dim=1)
    is_reverse2 = torch.cat([is_reverse, rev_is_reverse], dim=0)

    mask = ei2[0] != ei2[1]
    self_loop_mask = ~mask
    existing_loop_nodes = ei2[0, self_loop_mask]
    existing_loop_is_rev = is_reverse2[self_loop_mask]
    loop_index = torch.arange(num_nodes, dtype=ei2.dtype, device=ei2.device)
    loop_ei = loop_index.unsqueeze(0).repeat(2, 1)
    loop_is_reverse = torch.zeros(num_nodes, 1, dtype=torch.float)
    loop_is_reverse[existing_loop_nodes] = existing_loop_is_rev
    final_ei = torch.cat([ei2[:, mask], loop_ei], dim=1)
    final_is_reverse = torch.cat([is_reverse2[mask], loop_is_reverse], dim=0)
    return final_ei, final_is_reverse


@torch.no_grad()
def diagnose_is_reverse_bug(base_ei, base_ea, sample_target_idx, num_neighbors, batch_size=512):
    target_nodes = torch.unique(torch.cat([
        data.edge_index[0, sample_target_idx], data.edge_index[1, sample_target_idx]
    ]))
    base_graph = Data(x=data.x, edge_index=base_ei, edge_attr=base_ea, num_nodes=data.x.size(0))
    loader = NeighborLoader(base_graph, num_neighbors=num_neighbors, input_nodes=target_nodes,
                             batch_size=batch_size, shuffle=False)

    n_subgraphs, n_affected_subgraphs = 0, 0
    n_rows_checked, n_rows_mismatched = 0, 0
    structure_mismatch_warnings = 0

    for sub in loader:
        n_subgraphs += 1
        has_genuine_loop = (sub.edge_attr[:, IS_SELF_LOOP_COL] == 1).any().item()
        if not has_genuine_loop:
            continue
        n_affected_subgraphs += 1

        ei_buggy, is_rev_buggy = _is_reverse_buggy(sub.edge_index, sub.edge_attr, sub.num_nodes)
        ei_correct, is_rev_correct = _is_reverse_correct(sub.edge_index, sub.edge_attr, sub.num_nodes)

        if not torch.equal(ei_buggy, ei_correct):
            # Would mean our manual replica of add_remaining_self_loops doesn't
            # actually match the library's behavior on this PyG version -- if this
            # ever fires, the diagnostic's conclusion below isn't trustworthy, stop
            # and re-check add_remaining_self_loops' source for this installation.
            structure_mismatch_warnings += 1
            continue

        n_rows_checked += is_rev_buggy.size(0)
        n_rows_mismatched += (is_rev_buggy != is_rev_correct).sum().item()

    return {
        "subgraphs_checked": n_subgraphs,
        "subgraphs_with_genuine_self_loop": n_affected_subgraphs,
        "rows_checked": n_rows_checked,
        "rows_with_wrong_is_reverse": n_rows_mismatched,
        "structure_mismatch_warnings": structure_mismatch_warnings,
    }


DIAG_SAMPLE_SIZE = 5000
g = torch.Generator().manual_seed(RANDOM_SEED + 42)
diag_sample_idx = data.train_mask.nonzero(as_tuple=True)[0]
diag_sample_idx = diag_sample_idx[torch.randperm(diag_sample_idx.size(0), generator=g)[:DIAG_SAMPLE_SIZE]]

diag_result = diagnose_is_reverse_bug(train_base_ei, train_base_ea, diag_sample_idx, NUM_NEIGHBORS_INFER)
print(json.dumps(diag_result, indent=2))

if diag_result["structure_mismatch_warnings"] > 0:
    print("\n*** Manual replica of add_remaining_self_loops did not match the library's actual "
          "behavior on this PyG version for some subgraphs -- treat the numbers above as "
          "unreliable and flag this before drawing any conclusion. ***")
elif diag_result["subgraphs_with_genuine_self_loop"] == 0:
    print("\nNo genuine self-transactions turned up in this sample -- the bug is real but may "
          "not have materially affected best_model_pna.pt. Consider re-running with a larger "
          "DIAG_SAMPLE_SIZE before concluding it's a non-issue.")
else:
    pct_rows = 100 * diag_result["rows_with_wrong_is_reverse"] / max(diag_result["rows_checked"], 1)
    print(f"\n{diag_result['subgraphs_with_genuine_self_loop']:,} / {diag_result['subgraphs_checked']:,} "
          f"sampled subgraphs contained a genuine self-transaction, affecting "
          f"{diag_result['rows_with_wrong_is_reverse']:,} edge rows ({pct_rows:.4f}% of all rows checked) "
          f"with a flipped is_reverse bit. Use this rate to judge whether best_model_pna.pt is worth "
          f"revisiting -- this notebook doesn't retrain the classifier either way.")
